# Day 2: Data Cleaning, SQLite Star Schema & Analytics
Covers: NAV cleaning, transactions cleaning, scheme performance cleaning, SQLite star schema load, 10 analytical SQL queries, data dictionary.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from sqlalchemy import create_engine, text
from pathlib import Path

RAW = Path('../data/raw')
PROCESSED = Path('../data/processed')
PROCESSED.mkdir(parents=True, exist_ok=True)
DB_PATH = Path('../bluestock_mf.db')

print('Libraries loaded.')

Libraries loaded.


## 1. Clean NAV History (02_nav_history.csv)

In [2]:
nav_raw = pd.read_csv(RAW / '02_nav_history.csv')
print(f'Raw shape: {nav_raw.shape}')
print(nav_raw.dtypes)
nav_raw.head()

Raw shape: (46000, 3)
amfi_code      int64
date          object
nav          float64
dtype: object


,amfi_code,date,nav
0,119551,2022-01-03,54.3856
1,119551,2022-01-04,54.3474
2,119551,2022-01-05,54.6869
3,119551,2022-01-06,55.4550
4,119551,2022-01-07,55.3692


In [3]:
nav = nav_raw.copy()

# 1. Parse date
nav['date'] = pd.to_datetime(nav['date'], errors='coerce')
invalid_dates = nav['date'].isna().sum()
print(f'Invalid dates dropped: {invalid_dates}')
nav = nav.dropna(subset=['date'])

# 2. Validate NAV > 0
invalid_nav = (nav['nav'] <= 0).sum()
print(f'NAV <= 0 rows removed: {invalid_nav}')
nav = nav[nav['nav'] > 0]

# 3. Remove duplicates
dupes = nav.duplicated(subset=['amfi_code', 'date']).sum()
print(f'Duplicate rows removed: {dupes}')
nav = nav.drop_duplicates(subset=['amfi_code', 'date'])

# 4. Sort by amfi_code + date
nav = nav.sort_values(['amfi_code', 'date']).reset_index(drop=True)

# 5. Forward-fill missing NAV for holidays/weekends
# Build full calendar per fund and ffill
date_range = pd.date_range(nav['date'].min(), nav['date'].max(), freq='D')
funds = nav['amfi_code'].unique()
idx = pd.MultiIndex.from_product([funds, date_range], names=['amfi_code', 'date'])
nav_full = nav.set_index(['amfi_code', 'date']).reindex(idx)
nav_full['nav'] = nav_full.groupby('amfi_code')['nav'].ffill()
# Drop rows that are still NaN (before first trading day of a fund)
nav_full = nav_full.dropna(subset=['nav']).reset_index()

# 6. is_trading_day flag
original_dates = set(zip(nav_raw['amfi_code'], pd.to_datetime(nav_raw['date']).dt.date))
nav_full['is_trading_day'] = nav_full.apply(
    lambda r: (r['amfi_code'], r['date'].date()) in original_dates, axis=1
).astype(int)

print(f'Cleaned NAV shape: {nav_full.shape}')
nav_full.head(10)

Invalid dates dropped: 0
NAV <= 0 rows removed: 0
Duplicate rows removed: 0


Cleaned NAV shape: (64320, 4)


,amfi_code,date,nav,is_trading_day
0,100016,2022-01-03,520.4608,1
1,100016,2022-01-04,515.0971,1
2,100016,2022-01-05,521.7239,1
3,100016,2022-01-06,515.7880,1
4,100016,2022-01-07,515.1639,1
5,100016,2022-01-08,515.1639,0
6,100016,2022-01-09,515.1639,0
7,100016,2022-01-10,510.7136,1
8,100016,2022-01-11,513.5542,1
9,100016,2022-01-12,512.3195,1


In [4]:
nav_full.to_csv(PROCESSED / '02_nav_history_cleaned.csv', index=False)
print('Saved 02_nav_history_cleaned.csv')

Saved 02_nav_history_cleaned.csv


## 2. Clean Investor Transactions (08_investor_transactions.csv)

In [5]:
txn_raw = pd.read_csv(RAW / '08_investor_transactions.csv')
print(f'Raw shape: {txn_raw.shape}')
print(txn_raw.dtypes)
txn_raw.head()

Raw shape: (32778, 13)
investor_id            object
transaction_date       object
amfi_code               int64
transaction_type       object
amount_inr              int64
state                  object
city                   object
city_tier              object
age_group              object
gender                 object
annual_income_lakh    float64
payment_mode           object
kyc_status             object
dtype: object


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [6]:
txn = txn_raw.copy()

# 1. Fix date format
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'], errors='coerce')
bad_dates = txn['transaction_date'].isna().sum()
print(f'Bad transaction dates dropped: {bad_dates}')
txn = txn.dropna(subset=['transaction_date'])

# 2. Standardise transaction_type
VALID_TXN_TYPES = {'SIP', 'Lumpsum', 'Redemption'}
type_map = {
    'sip': 'SIP', 'systematic investment plan': 'SIP',
    'lumpsum': 'Lumpsum', 'lump sum': 'Lumpsum', 'one time': 'Lumpsum', 'onetime': 'Lumpsum',
    'redemption': 'Redemption', 'redeem': 'Redemption', 'withdrawal': 'Redemption'
}
txn['transaction_type'] = (
    txn['transaction_type']
    .str.strip()
    .apply(lambda x: type_map.get(x.lower(), x) if isinstance(x, str) else x)
)
invalid_types = ~txn['transaction_type'].isin(VALID_TXN_TYPES)
print(f'Unknown transaction_type values: {txn.loc[invalid_types, "transaction_type"].unique()}')
txn = txn[~invalid_types]

# 3. Validate amount > 0
bad_amount = (txn['amount_inr'] <= 0).sum()
print(f'Amount <= 0 rows removed: {bad_amount}')
txn = txn[txn['amount_inr'] > 0]

# 4. Check KYC status enum values
VALID_KYC = {'Verified', 'Pending', 'Rejected', 'KYC_Registered'}
print(f'KYC status values: {txn["kyc_status"].unique()}')
invalid_kyc = ~txn['kyc_status'].isin(VALID_KYC)
print(f'Invalid KYC rows: {invalid_kyc.sum()}')
# Standardise casing
txn['kyc_status'] = txn['kyc_status'].str.strip().str.title()

# 5. Remove duplicates
dupes = txn.duplicated().sum()
print(f'Duplicate rows removed: {dupes}')
txn = txn.drop_duplicates().reset_index(drop=True)

print(f'Cleaned transactions shape: {txn.shape}')
txn.head()

Bad transaction dates dropped: 0
Unknown transaction_type values: []
Amount <= 0 rows removed: 0
KYC status values: ['Verified' 'Pending']
Invalid KYC rows: 0
Duplicate rows removed: 0
Cleaned transactions shape: (32778, 13)


,investor_id,transaction_date,amfi_code,transaction_type,amount_inr,state,city,city_tier,age_group,gender,annual_income_lakh,payment_mode,kyc_status
0,INV003054,2024-01-01,119092,SIP,1834,Telangana,Hyderabad,T30,56+,Female,77.1,UPI,Verified
1,INV002952,2024-01-01,148567,Redemption,392882,Punjab,Amritsar,B30,18-25,Male,7.1,Cheque,Verified
2,INV003420,2024-01-01,118636,SIP,912,Haryana,Faridabad,B30,36-45,Male,47.2,Mandate,Verified
3,INV003436,2024-01-01,118634,SIP,1102,Maharashtra,Mumbai,T30,36-45,Female,54.4,Cheque,Pending
4,INV004691,2024-01-01,119094,Lumpsum,8682,Delhi,Noida,T30,26-35,Male,14.5,Net Banking,Pending


In [7]:
txn.to_csv(PROCESSED / '08_investor_transactions_cleaned.csv', index=False)
print('Saved 08_investor_transactions_cleaned.csv')

Saved 08_investor_transactions_cleaned.csv


## 3. Clean Scheme Performance (07_scheme_performance.csv)

In [8]:
perf_raw = pd.read_csv(RAW / '07_scheme_performance.csv')
print(f'Raw shape: {perf_raw.shape}')
perf_raw.head()

Raw shape: (40, 19)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low


In [9]:
perf = perf_raw.copy()

RETURN_COLS = ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct']
NUMERIC_COLS = RETURN_COLS + ['alpha', 'beta', 'sharpe_ratio', 'sortino_ratio',
                               'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct']

# 1. Validate all return values are numeric
for col in NUMERIC_COLS:
    perf[col] = pd.to_numeric(perf[col], errors='coerce')
    bad = perf[col].isna().sum()
    if bad:
        print(f'  {col}: {bad} non-numeric values coerced to NaN')

# 2. Flag return anomalies (returns outside -50% to +100%)
for col in RETURN_COLS:
    anomalies = perf[(perf[col] < -50) | (perf[col] > 100)]
    if len(anomalies):
        print(f'Anomaly in {col}: {anomalies[["scheme_name", col]].values}')
    else:
        print(f'{col}: No anomalies detected')

# 3. Check expense_ratio range (0.1% – 2.5%)
out_of_range = perf[(perf['expense_ratio_pct'] < 0.1) | (perf['expense_ratio_pct'] > 2.5)]
if len(out_of_range):
    print(f'\nExpense ratio out of 0.1–2.5% range:')
    print(out_of_range[['scheme_name', 'expense_ratio_pct']])
else:
    print('\nAll expense ratios in valid range (0.1%–2.5%)')

# 4. Flag (don't drop) anomaly rows
perf['anomaly_flag'] = (
    (perf[RETURN_COLS] < -50).any(axis=1) |
    (perf[RETURN_COLS] > 100).any(axis=1) |
    (perf['expense_ratio_pct'] < 0.1) |
    (perf['expense_ratio_pct'] > 2.5)
).astype(int)

print(f'\nCleaned scheme performance shape: {perf.shape}')
perf.head()

return_1yr_pct: No anomalies detected
return_3yr_pct: No anomalies detected
return_5yr_pct: No anomalies detected
benchmark_3yr_pct: No anomalies detected

All expense ratios in valid range (0.1%–2.5%)

Cleaned scheme performance shape: (40, 20)


,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,beta,sharpe_ratio,sortino_ratio,std_dev_ann_pct,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade,anomaly_flag
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,0.89,0.88,1.29,14.0,-21.70,14288,1.54,4,Moderate,0
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,0.87,0.81,1.29,14.0,-24.43,1231,0.66,3,Moderate,0
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,0.89,0.94,1.35,25.0,-13.35,19259,1.43,5,Very High,0
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,1.04,0.93,1.67,25.0,-24.78,36061,0.72,4,Very High,0
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,0.22,1.52,2.11,4.0,-2.30,24101,0.77,5,Low,0


In [10]:
perf.to_csv(PROCESSED / '07_scheme_performance_cleaned.csv', index=False)
print('Saved 07_scheme_performance_cleaned.csv')

Saved 07_scheme_performance_cleaned.csv


## 4. Clean & Save Remaining Raw Files

In [11]:
# Fund Master
fund_master = pd.read_csv(RAW / '01_fund_master.csv')
fund_master['launch_date'] = pd.to_datetime(fund_master['launch_date'], errors='coerce')
fund_master = fund_master.drop_duplicates(subset=['amfi_code'])
fund_master.to_csv(PROCESSED / '01_fund_master_cleaned.csv', index=False)
print(f'01_fund_master_cleaned: {fund_master.shape}')

# AUM by fund house
aum = pd.read_csv(RAW / '03_aum_by_fund_house.csv')
aum['date'] = pd.to_datetime(aum['date'], errors='coerce')
aum = aum.dropna(subset=['date'])
aum.to_csv(PROCESSED / '03_aum_by_fund_house_cleaned.csv', index=False)
print(f'03_aum_by_fund_house_cleaned: {aum.shape}')

# Monthly SIP inflows
sip = pd.read_csv(RAW / '04_monthly_sip_inflows.csv')
sip['month'] = pd.to_datetime(sip['month'], format='%Y-%m', errors='coerce')
sip['yoy_growth_pct'] = pd.to_numeric(sip['yoy_growth_pct'], errors='coerce')
sip.to_csv(PROCESSED / '04_monthly_sip_inflows_cleaned.csv', index=False)
print(f'04_monthly_sip_inflows_cleaned: {sip.shape}')

# Category inflows
cat = pd.read_csv(RAW / '05_category_inflows.csv')
cat.to_csv(PROCESSED / '05_category_inflows_cleaned.csv', index=False)
print(f'05_category_inflows_cleaned: {cat.shape}')

# Industry folio count
folio = pd.read_csv(RAW / '06_industry_folio_count.csv')
folio.to_csv(PROCESSED / '06_industry_folio_count_cleaned.csv', index=False)
print(f'06_industry_folio_count_cleaned: {folio.shape}')

# Portfolio holdings
holdings = pd.read_csv(RAW / '09_portfolio_holdings.csv')
holdings.to_csv(PROCESSED / '09_portfolio_holdings_cleaned.csv', index=False)
print(f'09_portfolio_holdings_cleaned: {holdings.shape}')

# Benchmark indices
bench = pd.read_csv(RAW / '10_benchmark_indices.csv')
bench.to_csv(PROCESSED / '10_benchmark_indices_cleaned.csv', index=False)
print(f'10_benchmark_indices_cleaned: {bench.shape}')

01_fund_master_cleaned: (40, 15)


03_aum_by_fund_house_cleaned: (90, 5)


04_monthly_sip_inflows_cleaned: (48, 6)
05_category_inflows_cleaned: (144, 3)
06_industry_folio_count_cleaned: (21, 6)
09_portfolio_holdings_cleaned: (322, 8)


10_benchmark_indices_cleaned: (8050, 3)


## 5. Create SQLite Star Schema

In [12]:
engine = create_engine(f'sqlite:///{DB_PATH}', echo=False)

schema_sql = '''
PRAGMA foreign_keys = ON;

-- Dimension: Fund
CREATE TABLE IF NOT EXISTS dim_fund (
    amfi_code       INTEGER PRIMARY KEY,
    fund_house      TEXT    NOT NULL,
    scheme_name     TEXT    NOT NULL,
    category        TEXT,
    sub_category    TEXT,
    plan            TEXT,
    launch_date     TEXT,
    benchmark       TEXT,
    expense_ratio   REAL,
    exit_load_pct   REAL,
    min_sip_amount  REAL,
    fund_manager    TEXT,
    risk_category   TEXT,
    sebi_category_code TEXT
);

-- Dimension: Date
CREATE TABLE IF NOT EXISTS dim_date (
    date_key    TEXT PRIMARY KEY,  -- YYYY-MM-DD
    year        INTEGER,
    quarter     INTEGER,
    month       INTEGER,
    month_name  TEXT,
    day         INTEGER,
    day_of_week INTEGER,
    week_of_year INTEGER,
    is_weekend  INTEGER,
    is_month_end INTEGER
);

-- Fact: NAV
CREATE TABLE IF NOT EXISTS fact_nav (
    nav_id          INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code       INTEGER NOT NULL REFERENCES dim_fund(amfi_code),
    date_key        TEXT    NOT NULL REFERENCES dim_date(date_key),
    nav             REAL    NOT NULL CHECK(nav > 0),
    is_trading_day  INTEGER DEFAULT 1
);

-- Fact: Investor Transactions
CREATE TABLE IF NOT EXISTS fact_transactions (
    txn_id          INTEGER PRIMARY KEY AUTOINCREMENT,
    investor_id     TEXT    NOT NULL,
    date_key        TEXT    NOT NULL REFERENCES dim_date(date_key),
    amfi_code       INTEGER NOT NULL REFERENCES dim_fund(amfi_code),
    transaction_type TEXT   NOT NULL CHECK(transaction_type IN ('SIP','Lumpsum','Redemption')),
    amount_inr      REAL    NOT NULL CHECK(amount_inr > 0),
    state           TEXT,
    city            TEXT,
    city_tier       TEXT,
    age_group       TEXT,
    gender          TEXT,
    annual_income_lakh REAL,
    payment_mode    TEXT,
    kyc_status      TEXT
);

-- Fact: Scheme Performance
CREATE TABLE IF NOT EXISTS fact_performance (
    perf_id         INTEGER PRIMARY KEY AUTOINCREMENT,
    amfi_code       INTEGER NOT NULL REFERENCES dim_fund(amfi_code),
    return_1yr_pct  REAL,
    return_3yr_pct  REAL,
    return_5yr_pct  REAL,
    benchmark_3yr_pct REAL,
    alpha           REAL,
    beta            REAL,
    sharpe_ratio    REAL,
    sortino_ratio   REAL,
    std_dev_ann_pct REAL,
    max_drawdown_pct REAL,
    aum_crore       REAL,
    expense_ratio_pct REAL,
    morningstar_rating INTEGER,
    risk_grade      TEXT,
    anomaly_flag    INTEGER DEFAULT 0
);

-- Fact: AUM by Fund House
CREATE TABLE IF NOT EXISTS fact_aum (
    aum_id          INTEGER PRIMARY KEY AUTOINCREMENT,
    date_key        TEXT    NOT NULL REFERENCES dim_date(date_key),
    fund_house      TEXT    NOT NULL,
    aum_lakh_crore  REAL,
    aum_crore       REAL,
    num_schemes     INTEGER
);
'''

with engine.connect() as conn:
    for stmt in schema_sql.strip().split(';'):
        stmt = stmt.strip()
        if stmt:
            conn.execute(text(stmt))
    conn.commit()

print('Star schema created in bluestock_mf.db')

Star schema created in bluestock_mf.db


## 6. Populate dim_date

In [13]:
all_dates = pd.date_range('2022-01-01', '2026-12-31', freq='D')
dim_date = pd.DataFrame({
    'date_key':     all_dates.strftime('%Y-%m-%d'),
    'year':         all_dates.year,
    'quarter':      all_dates.quarter,
    'month':        all_dates.month,
    'month_name':   all_dates.strftime('%B'),
    'day':          all_dates.day,
    'day_of_week':  all_dates.dayofweek,   # 0=Mon
    'week_of_year': all_dates.isocalendar().week.values,
    'is_weekend':   (all_dates.dayofweek >= 5).astype(int),
    'is_month_end': all_dates.is_month_end.astype(int)
})

dim_date.to_sql('dim_date', engine, if_exists='replace', index=False)
print(f'dim_date loaded: {len(dim_date)} rows')

dim_date loaded: 1826 rows


## 7. Load All Cleaned Datasets into SQLite

In [14]:
# dim_fund
fund_master = pd.read_csv(PROCESSED / '01_fund_master_cleaned.csv')
fund_master_db = fund_master.rename(columns={'expense_ratio_pct': 'expense_ratio'})[[
    'amfi_code','fund_house','scheme_name','category','sub_category','plan',
    'launch_date','benchmark','expense_ratio','exit_load_pct',
    'min_sip_amount','fund_manager','risk_category','sebi_category_code'
]]
fund_master_db.to_sql('dim_fund', engine, if_exists='replace', index=False)
print(f'dim_fund: {len(fund_master_db)} rows')

dim_fund: 40 rows


In [15]:
# fact_nav
nav_full = pd.read_csv(PROCESSED / '02_nav_history_cleaned.csv')
nav_full['date'] = pd.to_datetime(nav_full['date'])
nav_db = nav_full[['amfi_code','date','nav','is_trading_day']].copy()
nav_db['date_key'] = nav_db['date'].dt.strftime('%Y-%m-%d')
nav_db = nav_db.drop(columns=['date'])
nav_db.to_sql('fact_nav', engine, if_exists='replace', index=False)
print(f'fact_nav: {len(nav_db)} rows')

fact_nav: 64320 rows


In [16]:
# fact_transactions
txn = pd.read_csv(PROCESSED / '08_investor_transactions_cleaned.csv')
txn['transaction_date'] = pd.to_datetime(txn['transaction_date'])
txn_db = txn.copy()
txn_db['date_key'] = txn_db['transaction_date'].dt.strftime('%Y-%m-%d')
txn_db = txn_db.drop(columns=['transaction_date'])
txn_db.to_sql('fact_transactions', engine, if_exists='replace', index=False)
print(f'fact_transactions: {len(txn_db)} rows')

fact_transactions: 32778 rows


In [17]:
# fact_performance
perf = pd.read_csv(PROCESSED / '07_scheme_performance_cleaned.csv')
perf.to_sql('fact_performance', engine, if_exists='replace', index=False)
print(f'fact_performance: {len(perf)} rows')

fact_performance: 40 rows


In [18]:
# fact_aum
aum = pd.read_csv(PROCESSED / '03_aum_by_fund_house_cleaned.csv')
aum['date'] = pd.to_datetime(aum['date'])
aum_db = aum.copy()
aum_db['date_key'] = aum_db['date'].dt.strftime('%Y-%m-%d')
aum_db = aum_db.drop(columns=['date'])
aum_db.to_sql('fact_aum', engine, if_exists='replace', index=False)
print(f'fact_aum: {len(aum_db)} rows')

fact_aum: 90 rows


## 8. Verify Row Counts Match Source CSVs

In [19]:
tables = {
    'dim_fund':         fund_master_db,
    'fact_nav':         nav_db,
    'fact_transactions': txn_db,
    'fact_performance': perf,
    'fact_aum':         aum_db,
}

with engine.connect() as conn:
    print(f'{'Table':<25} {'Python rows':>12} {'SQLite rows':>12} {'Match':>8}')
    print('-' * 60)
    for table, df in tables.items():
        db_count = conn.execute(text(f'SELECT COUNT(*) FROM {table}')).scalar()
        match = '✓' if len(df) == db_count else '✗ MISMATCH'
        print(f'{table:<25} {len(df):>12} {db_count:>12} {match:>8}')

Table                      Python rows  SQLite rows    Match
------------------------------------------------------------
dim_fund                            40           40        ✓
fact_nav                         64320        64320        ✓
fact_transactions                32778        32778        ✓
fact_performance                    40           40        ✓
fact_aum                            90           90        ✓


## 9. Analytical SQL Queries

In [20]:
def run_query(title, sql):
    print(f'\n{'='*60}')
    print(f'Q: {title}')
    print('='*60)
    result = pd.read_sql_query(sql, engine)
    print(result.to_string(index=False))
    return result

In [21]:
# Q1: Top 5 funds by AUM (from scheme performance)
run_query('Top 5 Funds by AUM', """
SELECT f.scheme_name, f.fund_house, f.category, p.aum_crore
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
ORDER BY p.aum_crore DESC
LIMIT 5;
""")


Q: Top 5 Funds by AUM
                                          scheme_name        fund_house category  aum_crore
Mirae Asset Emerging Bluechip Fund - Regular - Growth    Mirae Asset MF   Equity      49046
        Kotak Emerging Equity Fund - Regular - Growth Kotak Mahindra MF   Equity      47469
       Nippon India Small Cap Fund - Regular - Growth   Nippon India MF   Equity      43630
           DSP Top 100 Equity Fund - Regular - Growth   DSP Mutual Fund   Equity      41828
                  UTI Mid Cap Fund - Regular - Growth   UTI Mutual Fund   Equity      41728


,scheme_name,fund_house,category,aum_crore
0,Mirae Asset Emerging Bluechip Fund - Regular -...,Mirae Asset MF,Equity,49046
1,Kotak Emerging Equity Fund - Regular - Growth,Kotak Mahindra MF,Equity,47469
2,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,Equity,43630
3,DSP Top 100 Equity Fund - Regular - Growth,DSP Mutual Fund,Equity,41828
4,UTI Mid Cap Fund - Regular - Growth,UTI Mutual Fund,Equity,41728


In [22]:
# Q2: Average NAV per month (across all funds, trading days only)
run_query('Average NAV per Month (Trading Days)', """
SELECT d.year, d.month, d.month_name,
       ROUND(AVG(n.nav), 4) AS avg_nav,
       COUNT(*) AS data_points
FROM fact_nav n
JOIN dim_date d ON n.date_key = d.date_key
WHERE n.is_trading_day = 1
GROUP BY d.year, d.month
ORDER BY d.year, d.month
LIMIT 24;
""")


Q: Average NAV per Month (Trading Days)
 year  month month_name  avg_nav  data_points
 2022      1    January 207.0614          840
 2022      2   February 207.7178          800
 2022      3      March 209.6926          920
 2022      4      April 211.8335          840
 2022      5        May 212.7315          880
 2022      6       June 213.8609          880
 2022      7       July 213.9561          840
 2022      8     August 215.6840          920
 2022      9  September 218.4943          880
 2022     10    October 219.5296          840
 2022     11   November 223.4707          880
 2022     12   December 226.7606          880
 2023      1    January 230.6712          880
 2023      2   February 233.8477          800
 2023      3      March 238.0096          920
 2023      4      April 240.6133          800
 2023      5        May 241.8929          920
 2023      6       June 244.6055          880
 2023      7       July 245.7807          840
 2023      8     August 247.7346       

,year,month,month_name,avg_nav,data_points
0,2022,1,January,207.0614,840
1,2022,2,February,207.7178,800
2,2022,3,March,209.6926,920
3,2022,4,April,211.8335,840
4,2022,5,May,212.7315,880
5,2022,6,June,213.8609,880
6,2022,7,July,213.9561,840
7,2022,8,August,215.6840,920
8,2022,9,September,218.4943,880
9,2022,10,October,219.5296,840


In [23]:
# Q3: SIP inflow YoY growth from monthly SIP data
sip = pd.read_csv(PROCESSED / '04_monthly_sip_inflows_cleaned.csv')
sip['month'] = pd.to_datetime(sip['month'])
sip.to_sql('fact_sip_inflows', engine, if_exists='replace', index=False)

run_query('SIP YoY Growth by Year', """
SELECT
    strftime('%Y', month) AS year,
    ROUND(SUM(sip_inflow_crore), 0) AS total_sip_crore,
    ROUND(AVG(yoy_growth_pct), 2) AS avg_yoy_growth_pct
FROM fact_sip_inflows
WHERE yoy_growth_pct IS NOT NULL
GROUP BY year
ORDER BY year;
""")


Q: SIP YoY Growth by Year
year  total_sip_crore  avg_yoy_growth_pct
2023         184763.0               23.49
2024         269781.0               45.69
2025         335740.0               25.19


,year,total_sip_crore,avg_yoy_growth_pct
0,2023,184763.0,23.49
1,2024,269781.0,45.69
2,2025,335740.0,25.19


In [24]:
# Q4: Total transaction volume by state
run_query('Total Transactions by State (Top 10)', """
SELECT state,
       COUNT(*) AS num_transactions,
       ROUND(SUM(amount_inr) / 1e7, 2) AS total_amount_crore
FROM fact_transactions
GROUP BY state
ORDER BY total_amount_crore DESC
LIMIT 10;
""")


Q: Total Transactions by State (Top 10)
         state  num_transactions  total_amount_crore
        Punjab              2965               31.58
    Tamil Nadu              2806               31.52
Madhya Pradesh              2931               30.83
     Rajasthan              2577               29.86
       Gujarat              2780               29.84
   West Bengal              2748               29.72
     Telangana              2718               29.02
         Delhi              2677               28.96
 Uttar Pradesh              2695               28.54
       Haryana              2736               27.96


,state,num_transactions,total_amount_crore
0,Punjab,2965,31.58
1,Tamil Nadu,2806,31.52
2,Madhya Pradesh,2931,30.83
3,Rajasthan,2577,29.86
4,Gujarat,2780,29.84
5,West Bengal,2748,29.72
6,Telangana,2718,29.02
7,Delhi,2677,28.96
8,Uttar Pradesh,2695,28.54
9,Haryana,2736,27.96


In [25]:
# Q5: Funds with expense_ratio < 1%
run_query('Funds with Expense Ratio < 1%', """
SELECT f.scheme_name, f.fund_house, f.category, p.expense_ratio_pct
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
WHERE p.expense_ratio_pct < 1.0
ORDER BY p.expense_ratio_pct ASC;
""")


Q: Funds with Expense Ratio < 1%
                                         scheme_name               fund_house category  expense_ratio_pct
Nippon India Gilt Securities Fund - Regular - Growth          Nippon India MF     Debt               0.55
        HDFC Short Term Debt Fund - Regular - Growth         HDFC Mutual Fund     Debt               0.56
                Kotak Liquid Fund - Regular - Growth        Kotak Mahindra MF     Debt               0.60
            SBI Bluechip Fund - Direct Plan - Growth          SBI Mutual Fund   Equity               0.66
           SBI Small Cap Fund - Direct Plan - Growth          SBI Mutual Fund   Equity               0.72
       Nippon India Large Cap Fund - Direct - Growth          Nippon India MF   Equity               0.72
            ICICI Pru Liquid Fund - Regular - Growth      ICICI Prudential MF     Debt               0.74
                Axis Bluechip Fund - Direct - Growth         Axis Mutual Fund   Equity               0.75
        SBI 

,scheme_name,fund_house,category,expense_ratio_pct
0,Nippon India Gilt Securities Fund - Regular - ...,Nippon India MF,Debt,0.55
1,HDFC Short Term Debt Fund - Regular - Growth,HDFC Mutual Fund,Debt,0.56
2,Kotak Liquid Fund - Regular - Growth,Kotak Mahindra MF,Debt,0.60
3,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Equity,0.66
4,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Equity,0.72
5,Nippon India Large Cap Fund - Direct - Growth,Nippon India MF,Equity,0.72
6,ICICI Pru Liquid Fund - Regular - Growth,ICICI Prudential MF,Debt,0.74
7,Axis Bluechip Fund - Direct - Growth,Axis Mutual Fund,Equity,0.75
8,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Debt,0.77
9,HDFC Mid-Cap Opportunities Fund - Direct - Growth,HDFC Mutual Fund,Equity,0.78


In [26]:
# Q6: Best performing funds by 5-year return
run_query('Top 5 Funds by 5-Year Return', """
SELECT f.scheme_name, f.fund_house, f.sub_category,
       p.return_5yr_pct, p.return_3yr_pct, p.return_1yr_pct
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
ORDER BY p.return_5yr_pct DESC
LIMIT 5;
""")


Q: Top 5 Funds by 5-Year Return
                                   scheme_name               fund_house sub_category  return_5yr_pct  return_3yr_pct  return_1yr_pct
        ABSL Small Cap Fund - Regular - Growth Aditya Birla Sun Life MF    Small Cap           23.80           22.38           24.93
        Axis Small Cap Fund - Regular - Growth         Axis Mutual Fund    Small Cap           22.62           20.98           21.97
Nippon India Small Cap Fund - Regular - Growth          Nippon India MF    Small Cap           21.88           20.15           21.30
     SBI Small Cap Fund - Direct Plan - Growth          SBI Mutual Fund    Small Cap           21.82           23.14           20.59
    SBI Small Cap Fund - Regular Plan - Growth          SBI Mutual Fund    Small Cap           20.67           23.39           24.56


,scheme_name,fund_house,sub_category,return_5yr_pct,return_3yr_pct,return_1yr_pct
0,ABSL Small Cap Fund - Regular - Growth,Aditya Birla Sun Life MF,Small Cap,23.80,22.38,24.93
1,Axis Small Cap Fund - Regular - Growth,Axis Mutual Fund,Small Cap,22.62,20.98,21.97
2,Nippon India Small Cap Fund - Regular - Growth,Nippon India MF,Small Cap,21.88,20.15,21.30
3,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,21.82,23.14,20.59
4,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,20.67,23.39,24.56


In [27]:
# Q7: Monthly SIP transaction count and average ticket size
run_query('Monthly SIP Transaction Count & Avg Ticket Size', """
SELECT d.year, d.month, d.month_name,
       COUNT(*) AS sip_count,
       ROUND(AVG(t.amount_inr), 0) AS avg_sip_amount
FROM fact_transactions t
JOIN dim_date d ON t.date_key = d.date_key
WHERE t.transaction_type = 'SIP'
GROUP BY d.year, d.month
ORDER BY d.year, d.month
LIMIT 18;
""")


Q: Monthly SIP Transaction Count & Avg Ticket Size
 year  month month_name  sip_count  avg_sip_amount
 2024      1    January       1146         11026.0
 2024      2   February       1154         10930.0
 2024      3      March       1177         10271.0
 2024      4      April       1186         11393.0
 2024      5        May       1155         11445.0
 2024      6       June       1155         11369.0
 2024      7       July       1235         10942.0
 2024      8     August       1154         10850.0
 2024      9  September       1142         10761.0
 2024     10    October       1167         10684.0
 2024     11   November       1108         11121.0
 2024     12   December       1179         10959.0
 2025      1    January       1234         10601.0
 2025      2   February       1041         10916.0
 2025      3      March       1196         11245.0
 2025      4      April       1086         11171.0
 2025      5        May       1201         11634.0


,year,month,month_name,sip_count,avg_sip_amount
0,2024,1,January,1146,11026.0
1,2024,2,February,1154,10930.0
2,2024,3,March,1177,10271.0
3,2024,4,April,1186,11393.0
4,2024,5,May,1155,11445.0
5,2024,6,June,1155,11369.0
6,2024,7,July,1235,10942.0
7,2024,8,August,1154,10850.0
8,2024,9,September,1142,10761.0
9,2024,10,October,1167,10684.0


In [28]:
# Q8: Fund house AUM growth between earliest and latest dates
run_query('Fund House AUM Growth (Earliest vs Latest)', """
WITH ranked AS (
    SELECT fund_house, aum_crore, date_key,
           RANK() OVER (PARTITION BY fund_house ORDER BY date_key ASC)  AS rk_asc,
           RANK() OVER (PARTITION BY fund_house ORDER BY date_key DESC) AS rk_desc
    FROM fact_aum
),
first_last AS (
    SELECT fund_house,
           MAX(CASE WHEN rk_asc  = 1 THEN aum_crore END) AS first_aum,
           MAX(CASE WHEN rk_desc = 1 THEN aum_crore END) AS last_aum
    FROM ranked
    GROUP BY fund_house
)
SELECT fund_house,
       ROUND(first_aum / 1e5, 2) AS first_aum_lakh_cr,
       ROUND(last_aum  / 1e5, 2) AS latest_aum_lakh_cr,
       ROUND((last_aum - first_aum) * 100.0 / first_aum, 1) AS growth_pct
FROM first_last
ORDER BY growth_pct DESC;
""")


Q: Fund House AUM Growth (Earliest vs Latest)
              fund_house  first_aum_lakh_cr  latest_aum_lakh_cr  growth_pct
          Mirae Asset MF               1.05                2.90       176.2
         Nippon India MF               2.70                7.00       159.3
     ICICI Prudential MF               4.65               10.74       131.0
       Kotak Mahindra MF               2.70                5.80       114.8
        HDFC Mutual Fund               4.35                9.30       113.8
         DSP Mutual Fund               1.10                2.30       109.1
         SBI Mutual Fund               6.05               12.50       106.6
         UTI Mutual Fund               2.30                4.10        78.3
Aditya Birla Sun Life MF               2.78                4.60        65.5
        Axis Mutual Fund               2.50                3.50        40.0


,fund_house,first_aum_lakh_cr,latest_aum_lakh_cr,growth_pct
0,Mirae Asset MF,1.05,2.90,176.2
1,Nippon India MF,2.70,7.00,159.3
2,ICICI Prudential MF,4.65,10.74,131.0
3,Kotak Mahindra MF,2.70,5.80,114.8
4,HDFC Mutual Fund,4.35,9.30,113.8
5,DSP Mutual Fund,1.10,2.30,109.1
6,SBI Mutual Fund,6.05,12.50,106.6
7,UTI Mutual Fund,2.30,4.10,78.3
8,Aditya Birla Sun Life MF,2.78,4.60,65.5
9,Axis Mutual Fund,2.50,3.50,40.0


In [29]:
# Q9: KYC status breakdown for Redemptions
run_query('KYC Status Breakdown for Redemptions', """
SELECT kyc_status,
       COUNT(*) AS redemption_count,
       ROUND(SUM(amount_inr) / 1e7, 2) AS total_crore
FROM fact_transactions
WHERE transaction_type = 'Redemption'
GROUP BY kyc_status
ORDER BY redemption_count DESC;
""")


Q: KYC Status Breakdown for Redemptions
kyc_status  redemption_count  total_crore
  Verified              4589       115.11
   Pending               378         9.34


,kyc_status,redemption_count,total_crore
0,Verified,4589,115.11
1,Pending,378,9.34


In [30]:
# Q10: Funds with alpha > 1 AND sharpe > 1 (quality screen)
run_query('Funds Passing Quality Screen (Alpha > 1 & Sharpe > 1)', """
SELECT f.scheme_name, f.sub_category,
       p.alpha, p.sharpe_ratio, p.return_3yr_pct, p.expense_ratio_pct,
       p.morningstar_rating
FROM fact_performance p
JOIN dim_fund f ON p.amfi_code = f.amfi_code
WHERE p.alpha > 1 AND p.sharpe_ratio > 1
ORDER BY p.sharpe_ratio DESC;
""")


Q: Funds Passing Quality Screen (Alpha > 1 & Sharpe > 1)
                                  scheme_name   sub_category  alpha  sharpe_ratio  return_3yr_pct  expense_ratio_pct  morningstar_rating
     ICICI Pru Liquid Fund - Regular - Growth         Liquid   1.85          7.68            7.68               0.74                   5
         Kotak Liquid Fund - Regular - Growth         Liquid   1.52          6.18            6.18               0.60                   3
          ABSL Liquid Fund - Regular - Growth         Liquid   1.18          5.14            5.14               0.79                   5
 HDFC Short Term Debt Fund - Regular - Growth Short Duration   1.98          1.84            7.37               0.56                   3
 SBI Magnum Gilt Fund - Regular Plan - Growth           Gilt   1.60          1.52            6.07               0.77                   5
Mirae Asset Large Cap Fund - Regular - Growth      Large Cap   1.62          1.06           14.81               1.46    

,scheme_name,sub_category,alpha,sharpe_ratio,return_3yr_pct,expense_ratio_pct,morningstar_rating
0,ICICI Pru Liquid Fund - Regular - Growth,Liquid,1.85,7.68,7.68,0.74,5
1,Kotak Liquid Fund - Regular - Growth,Liquid,1.52,6.18,6.18,0.60,3
2,ABSL Liquid Fund - Regular - Growth,Liquid,1.18,5.14,5.14,0.79,5
3,HDFC Short Term Debt Fund - Regular - Growth,Short Duration,1.98,1.84,7.37,0.56,3
4,SBI Magnum Gilt Fund - Regular Plan - Growth,Gilt,1.60,1.52,6.07,0.77,5
5,Mirae Asset Large Cap Fund - Regular - Growth,Large Cap,1.62,1.06,14.81,1.46,5


## 10. Summary: Row Count Verification

In [31]:
all_tables = ['dim_fund','dim_date','fact_nav','fact_transactions',
              'fact_performance','fact_aum','fact_sip_inflows']

with engine.connect() as conn:
    print('Final row counts in bluestock_mf.db:')
    print(f'{"Table":<30} {"Rows":>10}')
    print('-' * 42)
    for t in all_tables:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {t}')).scalar()
        print(f'{t:<30} {n:>10}')

print(f'\nDatabase saved to: {DB_PATH.resolve()}')

Final row counts in bluestock_mf.db:
Table                                Rows
------------------------------------------
dim_fund                               40
dim_date                             1826
fact_nav                            64320
fact_transactions                   32778
fact_performance                       40
fact_aum                               90
fact_sip_inflows                       48

Database saved to: D:\MutualFundAnalytics\bluestock_mf.db
